<a href="https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

**Lane:** Refresh / Content Opportunity Scoring

This notebook builds a simple machine-learning model for the same content-review problem used in Week 4.

The model uses observed **March 2026** signals to estimate whether a content item shows a **decline in sessions in April 2026**. The result is used for decision support: it helps rank items for human review, not automatic content changes.

The Week-4 baseline and the model are evaluated on the **same held-out client groups** and with the **same metric: ROC-AUC**.


## Setup

The FlyRank Internship Warehouse is accessed with the Hugging Face token stored in Colab Secrets as `HF_TOKEN`.


In [8]:
!pip -q install duckdb scikit-learn pandas

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.inspection import permutation_importance

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"{BASE}/fact_content_daily_performance/**/*.parquet"

print("Connected to FlyRank Internship Warehouse")


Connected to FlyRank Internship Warehouse


# 1. Method choice and why

I use a **Random Forest classifier**.

Why it fits this lane:

- The task needs a probability or score that can rank content items for review.
- The input signals may have non-linear relationships.
- Random Forest can combine several observed signals without assuming a straight-line relationship.
- The dataset uses a small, understandable feature set.
- I compare the model with the Week-4 baseline instead of assuming that a more complex method is automatically better.

The model is kept simple. The goal is useful decision support, not complexity.


In [9]:
# Build one row per content item for March and April.
# March signals are features. April performance is used only to create the future outcome.

query = f'''
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(ga4_sessions) AS sessions,
        SUM(ga4_engaged_sessions) AS engaged_sessions
    FROM read_parquet('{TABLE}', hive_partitioning=1)
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
      AND ga4_sessions > 0
    GROUP BY 1, 2
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(ga4_sessions) AS april_sessions
    FROM read_parquet('{TABLE}', hive_partitioning=1)
    WHERE month = '2026-04'
      AND ga4_data_available IS TRUE
      AND ga4_sessions > 0
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions,
    m.clicks,
    m.sessions,
    m.engaged_sessions,
    a.april_sessions
FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id
'''

df = con.sql(query).df()

# Observed March feature
df["engagement_rate"] = (
    df["engaged_sessions"] / df["sessions"]
).clip(0, 1)

# Future outcome:
# 1 = April sessions are lower than March sessions
# 0 = April sessions are equal or higher
df["target_decline"] = (
    df["april_sessions"] < df["sessions"]
).astype(int)

print("Rows:", len(df))
print("Positive decline rate:", round(df["target_decline"].mean(), 3))
df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 53298
Positive decline rate: 0.431


,client_hash_id,content_hash_id,impressions,clicks,sessions,engaged_sessions,april_sessions,engagement_rate,target_decline
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,458.0,2.0,14.0,0.0,13.0,0.000000,1
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3943.0,23.0,54.0,1.0,49.0,0.018519,1
2,client_65de48885f4ef01b,content_3c286ded8bd68120,2180.0,15.0,30.0,2.0,16.0,0.066667,1
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,503.0,8.0,23.0,1.0,19.0,0.043478,1
4,client_65de48885f4ef01b,content_ff867882e604fa96,24.0,0.0,2.0,0.0,10.0,0.000000,0


# 2. Split design

I use a **grouped validation split by pseudonymized client**.

Why this is more honest:

- Content from the same client can have similar traffic patterns.
- A random row split could place related client patterns in both training and test data.
- Grouping keeps each client entirely in either training or test data.

The model sees March signals during training. The target is based on later April performance, so the future outcome is not used as a feature.


In [10]:
features = [
    "impressions",
    "clicks",
    "sessions",
    "engaged_sessions",
    "engagement_rate"
]

X = df[features].copy()
y = df["target_decline"].copy()
groups = df["client_hash_id"].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients.intersection(test_clients)))
print("Train positive rate:", round(y_train.mean(), 3))
print("Test positive rate:", round(y_test.mean(), 3))


Train rows: 43960
Test rows: 9338
Train clients: 25
Test clients: 9
Client overlap: 0
Train positive rate: 0.392
Test positive rate: 0.617


# 3. Train + compare vs my baseline

The Week-4 baseline is recreated using the same March inputs:

**Baseline score = 0.6 × normalized impressions + 0.4 × inverse engagement rate**

Higher scores mean higher review priority.

For this notebook, both the baseline and the Random Forest are evaluated on the same held-out client groups using **ROC-AUC** against the April decline outcome.


In [11]:
# Recreate the Week-4 baseline score using March signals only.

test_data = df.iloc[test_idx].copy()

max_train_impressions = X_train["impressions"].max()

test_data["visibility_score"] = (
    test_data["impressions"] / max_train_impressions
).clip(0, 1)

test_data["low_engagement_score"] = (
    1 - test_data["engagement_rate"]
).clip(0, 1)

test_data["baseline_score"] = (
    0.6 * test_data["visibility_score"] +
    0.4 * test_data["low_engagement_score"]
)

baseline_auc = roc_auc_score(
    y_test,
    test_data["baseline_score"]
)

# Train the model
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_prob = model.predict_proba(X_test)[:, 1]
model_auc = roc_auc_score(y_test, model_prob)

comparison = pd.DataFrame({
    "Method": ["Week-4 baseline score", "Random Forest model"],
    "ROC-AUC": [baseline_auc, model_auc]
})

comparison["ROC-AUC"] = comparison["ROC-AUC"].round(3)

print(comparison.to_string(index=False))


               Method  ROC-AUC
Week-4 baseline score    0.586
  Random Forest model    0.918


### Comparison note

The comparison above is the main result. A higher ROC-AUC means the method ranks content items with a later decline above non-declining items more effectively.

The result should be interpreted as measured performance on this held-out split. It does not prove that refreshing content will cause recovery.


In [12]:
winner = comparison.loc[comparison["ROC-AUC"].idxmax(), "Method"]
difference = abs(model_auc - baseline_auc)

print("Better method on this split:", winner)
print("Absolute ROC-AUC difference:", round(difference, 3))

if model_auc > baseline_auc:
    print("Observation: the model improved ranking performance over the Week-4 baseline on this held-out split.")
else:
    print("Observation: the Week-4 baseline matched or outperformed the model on this held-out split. Extra complexity was not rewarded.")


Better method on this split: Random Forest model
Absolute ROC-AUC difference: 0.332
Observation: the model improved ranking performance over the Week-4 baseline on this held-out split.


# 4. Errors and interpretation

A short error analysis checks which items are difficult for the model.

I inspect:

- **False positives:** the model predicted decline, but April sessions did not decline.
- **False negatives:** the model did not predict decline, but April sessions declined.

These errors matter because both types can affect the human review queue.


In [13]:
error_df = test_data[
    [
        "impressions",
        "clicks",
        "sessions",
        "engaged_sessions",
        "engagement_rate",
        "target_decline"
    ]
].copy()

error_df["predicted_probability"] = model_prob
error_df["predicted_decline"] = (
    error_df["predicted_probability"] >= 0.5
).astype(int)

false_positives = error_df[
    (error_df["predicted_decline"] == 1) &
    (error_df["target_decline"] == 0)
]

false_negatives = error_df[
    (error_df["predicted_decline"] == 0) &
    (error_df["target_decline"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
display(false_positives.head(5))

print("\nExample false negatives:")
display(false_negatives.head(5))


False positives: 572
False negatives: 1203

Example false positives:


,impressions,clicks,sessions,engaged_sessions,engagement_rate,target_decline,predicted_probability,predicted_decline
13,227.0,7.0,12.0,1.0,0.083333,0,0.593399,1
267,399.0,9.0,21.0,3.0,0.142857,0,0.671498,1
270,22.0,2.0,6.0,0.0,0.000000,0,0.634371,1
281,466.0,8.0,9.0,1.0,0.111111,0,0.561484,1
292,285.0,1.0,7.0,0.0,0.000000,0,0.543470,1



Example false negatives:


,impressions,clicks,sessions,engaged_sessions,engagement_rate,target_decline,predicted_probability,predicted_decline
259,84.0,3.0,3.0,0.0,0.0,1,0.429398,0
262,441.0,3.0,9.0,0.0,0.0,1,0.491541,0
263,3170.0,10.0,16.0,0.0,0.0,1,0.442555,0
269,1124.0,9.0,12.0,0.0,0.0,1,0.427719,0
271,584.0,7.0,9.0,0.0,0.0,1,0.460592,0


### Feature interpretation

Permutation importance measures how much the model's ROC-AUC changes when one feature is shuffled.

This is an interpretation tool, not proof of causation. A high importance means the model relied more on that feature for prediction on the held-out data.


In [14]:
importance = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": features,
    "importance_mean": importance.importances_mean
}).sort_values("importance_mean", ascending=False)

importance_df["importance_mean"] = importance_df["importance_mean"].round(4)

print(importance_df.to_string(index=False))


         feature  importance_mean
        sessions           0.4357
          clicks           0.0205
     impressions           0.0127
 engagement_rate           0.0013
engaged_sessions           0.0006


### Error summary

The model can be wrong when observed March signals do not fully capture what changes in April.

Possible reasons include:

- traffic changes caused by external factors,
- content intent changing over time,
- differences between clients,
- limited history in the selected feature set.

Therefore, the output should remain a **decision-support ranking for human review**, not an automatic refresh decision.


In [15]:
print("Classification report at a 0.50 probability threshold:")
print(classification_report(y_test, (model_prob >= 0.5).astype(int), digits=3))

print("\nInterpretation summary:")
top_feature = importance_df.iloc[0]["feature"]
print(f"- The most important measured feature on this split was: {top_feature}.")
print(f"- False positives: {len(false_positives)}.")
print(f"- False negatives: {len(false_negatives)}.")
print("- The model output is directional decision support, not a causal claim.")


Classification report at a 0.50 probability threshold:
              precision    recall  f1-score   support

           0      0.714     0.840     0.772      3580
           1      0.888     0.791     0.837      5758

    accuracy                          0.810      9338
   macro avg      0.801     0.816     0.805      9338
weighted avg      0.822     0.810     0.812      9338


Interpretation summary:
- The most important measured feature on this split was: sessions.
- False positives: 572.
- False negatives: 1203.
- The model output is directional decision support, not a causal claim.


# Self-check

- [x] Every section above includes markdown reasoning and code.
- [x] The model is compared with the Week-4 baseline on the same held-out split.
- [x] The validation split is grouped by pseudonymized client.
- [x] March observed signals are features; April performance is the future outcome.
- [x] ROC-AUC is used for both methods.
- [x] Feature interpretation and error analysis are included.
- [x] No client names, URLs, or private queries are shown.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.

Before submitting: run **Runtime → Run all**, check the printed numbers, then commit the executed notebook as:

`work/notebooks/w05_model.ipynb`
